In [ ]:
import os

In [2]:
%pwd

'/home/abdulrahman/End-to-End-Text-Summarizer-NLP/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/home/abdulrahman/End-to-End-Text-Summarizer-NLP'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: str  

    num_train_epochs: int

    per_device_train_batch_size: int
    per_device_eval_batch_size: int
    gradient_accumulation_steps: int

    learning_rate: float
    weight_decay: float

    warmup_steps: int

    logging_steps: int

    evaluation_strategy: str
    eval_steps: int

    save_strategy: str
    save_steps: int
    save_total_limit: int

    predict_with_generate: bool
    generation_max_length: int
    generation_num_beams: int

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt=config.model_ckpt,

            num_train_epochs=params.num_train_epochs,

            per_device_train_batch_size=params.per_device_train_batch_size,
            per_device_eval_batch_size=params.per_device_eval_batch_size,
            gradient_accumulation_steps=params.gradient_accumulation_steps,

            learning_rate=params.learning_rate,
            weight_decay=params.weight_decay,

            warmup_steps=params.warmup_steps,

            logging_steps=params.logging_steps,

            evaluation_strategy=params.evaluation_strategy,
            eval_steps=params.eval_steps,

            save_strategy=params.save_strategy,
            save_steps=params.save_steps,
            save_total_limit=params.save_total_limit,

            predict_with_generate=params.predict_with_generate,
            generation_max_length=params.generation_max_length,
            generation_num_beams=params.generation_num_beams
        )

        return model_trainer_config

In [8]:
import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

from datasets import load_from_disk

/home/abdulrahman/anaconda3/envs/textS/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-04-30 22:10:59,080: INFO: config: PyTorch version 2.4.1 available.]


In [ ]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config


    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"

        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)

        model = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_ckpt
        ).to(device)

        seq2seq_data_collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            model=model
        )

        # Load transformed/tokenized dataset
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = Seq2SeqTrainingArguments(
            output_dir=self.config.root_dir,

            num_train_epochs=self.config.num_train_epochs,

            per_device_train_batch_size=self.config.per_device_train_batch_size,
            per_device_eval_batch_size=self.config.per_device_eval_batch_size,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,

            learning_rate=self.config.learning_rate,
            weight_decay=self.config.weight_decay,

            warmup_steps=self.config.warmup_steps,

            logging_steps=self.config.logging_steps,

            evaluation_strategy=self.config.evaluation_strategy,
            eval_steps=self.config.eval_steps,

            save_strategy=self.config.save_strategy,
            save_steps=self.config.save_steps,
            save_total_limit=self.config.save_total_limit,

            predict_with_generate=self.config.predict_with_generate,
            generation_max_length=self.config.generation_max_length,
            generation_num_beams=self.config.generation_num_beams,

            fp16=torch.cuda.is_available(),
            report_to="none"
        )

        trainer = Seq2SeqTrainer(
            model=model,
            args=trainer_args,
            tokenizer=tokenizer,
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["train"],
            eval_dataset=dataset_samsum_pt["validation"]
        )

        trainer.train()

        # Save model and tokenizer 
        save_path = os.path.join(self.config.root_dir, "distilbart-samsum-model")

        trainer.save_model(save_path)
        tokenizer.save_pretrained(save_path)

In [ ]:
try:
    config = ConfigurationManager()

    model_trainer_config = config.get_model_trainer_config()

    model_trainer = ModelTrainer(config=model_trainer_config)

    model_trainer.train()

except Exception as e:
    raise e

[2026-04-30 22:11:27,117: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-04-30 22:11:27,139: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-30 22:11:27,144: INFO: common: created directory at: artifacts]


[2026-04-30 22:11:27,150: INFO: common: created directory at: artifacts/model_trainer]
